In [1]:
# ===== Task 22: Fix Data Types (Pandas) =====
import pandas as pd
import numpy as np
from IPython.display import display

# ---------- 1. Create and load the dataset (raw text, like a messy real-world CSV) ----------
rng = np.random.default_rng(22)
n = 25
names = ['Aarav', 'Diya', 'Rohan', 'Sneha', 'Karthik', 'Priya', 'Vikram', 'Ananya', 'Arjun', 'Meera',
         'Rahul', 'Kavya', 'Nikhil', 'Isha', 'Suresh', 'Lakshmi', 'Manoj', 'Divya', 'Harsh', 'Pooja',
         'Varun', 'Neha', 'Gautam', 'Swati', 'Ajay']

age = [str(a) for a in rng.integers(22, 58, n)]
age[3], age[11] = 'unknown', 'not given'                              # bad entries inside a numeric column
salary = [f'{s:,}' for s in rng.integers(25000, 120000, n)]           # '52,000' is text, not a number
bonus = [f'{b}%' for b in rng.integers(5, 20, n)]                     # '10%' is text
dates = (pd.Timestamp('2018-01-01') + pd.to_timedelta(rng.integers(0, 2500, n), 'D')).strftime('%Y-%m-%d').tolist()
dates[7] = 'Unknown'                                                  # invalid date
rating = [str(r) for r in rng.choice([3.0, 3.5, 4.0, 4.5, 5.0], n)]
rating[5] = 'not rated'

raw = pd.DataFrame({
    'employee_id': np.arange(101, 101 + n),
    'name': names,
    'department': rng.choice(['HR', 'IT', 'Finance', 'Sales', 'Marketing'], n),
    'age': age,
    'salary': salary,
    'bonus_pct': bonus,
    'join_date': dates,
    'is_active': rng.choice(['Yes', 'No'], n, p=[0.8, 0.2]),
    'rating': rating,
    'experience_years': rng.integers(1, 15, n).astype(float)})

raw.to_csv('employees_raw.csv', index=False)
df = pd.read_csv('employees_raw.csv')
print('Total records:', len(df))
display(df.head(8))

# ---------- Part A: Original data types ----------
print('\nORIGINAL DATA TYPES')
print(df.dtypes)
print()
df.info()

# A2. Which columns have the WRONG type?
orig_dtypes = df.dtypes.astype(str)
should_be = {
    'employee_id': 'int (already correct)',
    'name': 'text (already correct)',
    'department': 'category',
    'age': 'integer number',
    'salary': 'number',
    'bonus_pct': 'number',
    'join_date': 'datetime',
    'is_active': 'bool (True/False)',
    'rating': 'float number',
    'experience_years': 'integer number'}
check = pd.DataFrame({'loaded_as': orig_dtypes, 'should_be': pd.Series(should_be)})
display(check)

# A3. What goes wrong with the wrong types?
print('1. Salary sorted as TEXT (wrong order):', df['salary'].sort_values().head(4).tolist())
print('   -> "100,000" comes before "25,000" because text is compared character by character.')

try:
    df['age'].mean()
except Exception as e:
    print('\n2. age.mean() fails on text ->', type(e).__name__)

try:
    df['join_date'].dt.year
except Exception as e:
    print('\n3. join_date.dt.year fails on text ->', type(e).__name__)

before_memory = df.memory_usage(deep=True).sum()
print('\n4. Memory used before conversion:', before_memory, 'bytes')

# ---------- Part B: Converting the columns (on a copy) ----------
df_fixed = df.copy()
print('\nWorking on a copy.')

# B1. age: text to integer with pd.to_numeric(errors='coerce')
# Why: the column holds numbers plus bad entries ('unknown', 'not given'). errors='coerce' turns bad entries
# into NaN instead of crashing. Int64 (capital I) is an integer type that allows missing values.
df_fixed['age'] = pd.to_numeric(df_fixed['age'], errors='coerce').astype('Int64')
print('age ->', df_fixed['age'].dtype, '| bad entries became NaN:', int(df_fixed['age'].isnull().sum()))

# B2. salary: remove the commas, then convert to a number
# Why: '52,000' contains a comma, so it cannot be converted directly. str.replace cleans the text first.
df_fixed['salary'] = pd.to_numeric(df_fixed['salary'].str.replace(',', '', regex=False))
print('salary ->', df_fixed['salary'].dtype)

# B3. bonus_pct: remove the % sign, then convert to a number
# Why: '10%' is text. After removing '%', it becomes a real number we can calculate with.
df_fixed['bonus_pct'] = pd.to_numeric(df_fixed['bonus_pct'].str.replace('%', '', regex=False))
print('bonus_pct ->', df_fixed['bonus_pct'].dtype)

# B4. join_date: text to datetime with pd.to_datetime(errors='coerce')
# Why: real dates allow year/month extraction, date differences and time-order sorting. Invalid dates become NaT.
df_fixed['join_date'] = pd.to_datetime(df_fixed['join_date'], errors='coerce')
print('join_date ->', df_fixed['join_date'].dtype, '| invalid dates became NaT:', int(df_fixed['join_date'].isnull().sum()))

# B5. is_active: 'Yes'/'No' to True/False with map()
# Why: a boolean column is smaller, clearer and works directly in filters.
df_fixed['is_active'] = df_fixed['is_active'].map({'Yes': True, 'No': False})
print('is_active ->', df_fixed['is_active'].dtype)

# B6. rating: text to float with pd.to_numeric(errors='coerce')
# Why: the column contains a 'not rated' entry, which stopped Pandas from reading it as numbers.
df_fixed['rating'] = pd.to_numeric(df_fixed['rating'], errors='coerce')
print('rating ->', df_fixed['rating'].dtype, '| bad entries became NaN:', int(df_fixed['rating'].isnull().sum()))

# B7. department: text to category with astype('category')
# Why: only 5 repeated values. The category type stores each value once, saving memory and speeding up grouping.
df_fixed['department'] = df_fixed['department'].astype('category')
print('department ->', df_fixed['department'].dtype, '| categories:', list(df_fixed['department'].cat.categories))

# B8. experience_years: float to integer with astype(int)
# Why: years of experience are whole numbers (5, not 5.0). No missing values, so a plain int works.
df_fixed['experience_years'] = df_fixed['experience_years'].astype(int)
print('experience_years ->', df_fixed['experience_years'].dtype)

# ---------- Part C: Check the result ----------
# C1. Original vs corrected data types
comparison = pd.DataFrame({'original': orig_dtypes, 'corrected': df_fixed.dtypes.astype(str)})
comparison['changed'] = comparison['original'] != comparison['corrected']
display(comparison)
df_fixed.info()

# C2. The problems are now fixed
print('\n1. Salary sorted as NUMBERS (correct):', df_fixed['salary'].sort_values().head(4).tolist())
print('2. Average age:', round(df_fixed['age'].mean(), 1))
print('3. Average salary:', round(df_fixed['salary'].mean()))
print('4. Employees per joining year:')
display(df_fixed['join_date'].dt.year.astype('Int64').value_counts().sort_index())
print('5. Active employees:', int(df_fixed['is_active'].sum()), 'of', len(df_fixed))
print('6. Average salary by department:')
display(df_fixed.groupby('department', observed=True)['salary'].mean().round(0))
print('Memory before:', before_memory, 'bytes | after:', df_fixed.memory_usage(deep=True).sum(), 'bytes')

# C3. Cleaned dataset
display(df_fixed.head(10))
df_fixed.to_csv('employees_fixed.csv', index=False)
print('Saved employees_fixed.csv')

# ---------- Summary: conversions and reasons ----------
cols = ['age', 'salary', 'bonus_pct', 'join_date', 'is_active', 'rating', 'department', 'experience_years']
summary = pd.DataFrame({
    'column': cols,
    'from': [orig_dtypes[c] for c in cols],
    'to': ['Int64', 'number', 'number', 'datetime', 'bool', 'float', 'category', 'int'],
    'method': ["to_numeric(errors='coerce')", "str.replace(',') + to_numeric", "str.replace('%') + to_numeric",
               "to_datetime(errors='coerce')", "map({'Yes': True, 'No': False})", "to_numeric(errors='coerce')",
               "astype('category')", 'astype(int)'],
    'reason': ['Numbers must be numeric to calculate; bad values become NaN', 'Comma text cannot be summed or sorted correctly',
               '% sign blocks calculations', 'Enables year/month extraction and time maths',
               'Cleaner and smaller than Yes/No text', "'not rated' blocked numeric reading",
               'Few repeated values: saves memory', 'Whole numbers need no decimals']})
display(summary)

# ---------- Observations ----------
print('OBSERVATIONS')
print('-' * 50)
print(f"1. {int(comparison['changed'].sum())} of {len(comparison)} columns had a data type that needed fixing.")
print(f"2. Invalid entries were safely turned into missing values: {int(df_fixed['age'].isnull().sum())} in age, {int(df_fixed['rating'].isnull().sum())} in rating, {int(df_fixed['join_date'].isnull().sum())} in join_date.")
print('3. With correct types, sorting, averages, date calculations and grouping all work properly.')

Total records: 25


,employee_id,name,department,age,salary,bonus_pct,join_date,is_active,rating,experience_years
0,101,Aarav,HR,49,"70,346",18%,2018-08-06,Yes,5.0,8.0
1,102,Diya,HR,35,"111,492",11%,2018-10-21,No,4.5,1.0
2,103,Rohan,IT,45,"56,308",19%,2019-04-10,No,4.0,6.0
3,104,Sneha,IT,unknown,"56,058",17%,2018-01-25,Yes,5.0,4.0
4,105,Karthik,Finance,55,"45,566",9%,2021-10-10,Yes,3.5,8.0
5,106,Priya,Marketing,25,"69,522",12%,2022-03-06,Yes,not rated,8.0
6,107,Vikram,Sales,31,"100,716",15%,2022-07-06,Yes,4.0,11.0
7,108,Ananya,Sales,45,"58,657",16%,Unknown,Yes,4.0,6.0



ORIGINAL DATA TYPES
employee_id           int64
name                 object
department           object
age                  object
salary               object
bonus_pct            object
join_date            object
is_active            object
rating               object
experience_years    float64
dtype: object

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   employee_id       25 non-null     int64  
 1   name              25 non-null     object 
 2   department        25 non-null     object 
 3   age               25 non-null     object 
 4   salary            25 non-null     object 
 5   bonus_pct         25 non-null     object 
 6   join_date         25 non-null     object 
 7   is_active         25 non-null     object 
 8   rating            25 non-null     object 
 9   experience_years  25 non-null     float64
dtypes: float64(1

,loaded_as,should_be
employee_id,int64,int (already correct)
name,object,text (already correct)
department,object,category
age,object,integer number
salary,object,number
bonus_pct,object,number
join_date,object,datetime
is_active,object,bool (True/False)
rating,object,float number
experience_years,float64,integer number


1. Salary sorted as TEXT (wrong order): ['100,716', '111,492', '111,910', '31,339']
   -> "100,000" comes before "25,000" because text is compared character by character.

2. age.mean() fails on text -> TypeError

3. join_date.dt.year fails on text -> AttributeError

4. Memory used before conversion: 11260 bytes

Working on a copy.
age -> Int64 | bad entries became NaN: 2
salary -> int64
bonus_pct -> int64
join_date -> datetime64[ns] | invalid dates became NaT: 1
is_active -> bool
rating -> float64 | bad entries became NaN: 1
department -> category | categories: ['Finance', 'HR', 'IT', 'Marketing', 'Sales']
experience_years -> int64


...

Memory before: 11260 bytes | after: 3404 bytes


,employee_id,name,department,age,salary,bonus_pct,join_date,is_active,rating,experience_years
0,101,Aarav,HR,49,70346,18,2018-08-06,True,5.0,8
1,102,Diya,HR,35,111492,11,2018-10-21,False,4.5,1
2,103,Rohan,IT,45,56308,19,2019-04-10,False,4.0,6
3,104,Sneha,IT,<NA>,56058,17,2018-01-25,True,5.0,4
4,105,Karthik,Finance,55,45566,9,2021-10-10,True,3.5,8
5,106,Priya,Marketing,25,69522,12,2022-03-06,True,NaN,8
6,107,Vikram,Sales,31,100716,15,2022-07-06,True,4.0,11
7,108,Ananya,Sales,45,58657,16,NaT,True,4.0,6
8,109,Arjun,Marketing,22,64985,9,2018-03-07,True,5.0,13
9,110,Meera,Sales,38,85869,13,2020-11-17,True,5.0,6


Saved employees_fixed.csv


,column,from,to,method,reason
0,age,object,Int64,to_numeric(errors='coerce'),Numbers must be numeric to calculate; bad valu...
1,salary,object,number,"str.replace(',') + to_numeric",Comma text cannot be summed or sorted correctly
2,bonus_pct,object,number,str.replace('%') + to_numeric,% sign blocks calculations
3,join_date,object,datetime,to_datetime(errors='coerce'),Enables year/month extraction and time maths
4,is_active,object,bool,"map({'Yes': True, 'No': False})",Cleaner and smaller than Yes/No text
5,rating,object,float,to_numeric(errors='coerce'),'not rated' blocked numeric reading
6,department,object,category,astype('category'),Few repeated values: saves memory
7,experience_years,float64,int,astype(int),Whole numbers need no decimals


OBSERVATIONS
--------------------------------------------------
1. 8 of 10 columns had a data type that needed fixing.
2. Invalid entries were safely turned into missing values: 2 in age, 1 in rating, 1 in join_date.
3. With correct types, sorting, averages, date calculations and grouping all work properly.
